This notebook:
- Reads input.json files from `Input_jsons/` and images from `dataset/images/`.
- Detects error bars using improved color-robust scanning.
- Writes predictions to `Error_bar_prediction/` in the required format.
- Evaluates predictions vs `Error_bar_groundTruth/` with accuracy + confusion matrix.


In [1]:
# Install dependencies
%pip install numpy Pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\sulta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json
import math
import numpy as np
from PIL import Image

In [3]:
# Paths
ROOT_DIR = Path.cwd().parent  # BarSight/
INPUT_DIR = ROOT_DIR / "Input_jsons"
IMAGES_DIR = ROOT_DIR / "dataset" / "images"
PRED_DIR = ROOT_DIR / "Error_bar_prediction"
GT_DIR = ROOT_DIR / "Error_bar_groundTruth"

PRED_DIR.mkdir(parents=True, exist_ok=True)

# Detection parameters (tweak if needed)
COLOR_TOL = 38.0      # color distance tolerance
BG_TOL = 18.0         # min distance from background to count as line
GAP_LIMIT = 3         # allowed gaps in scan (pixels)
WINDOW = 2            # horizontal scan window (pixels)
MIN_HITS = 1          # minimum matching pixels in window

print("Input files:", len(list(INPUT_DIR.glob("*.json"))))
print("Images:", len(list(IMAGES_DIR.glob("*.png"))))

Input files: 150
Images: 150


In [4]:
def _median_color(pixels):
    if pixels.size == 0:
        return np.array([255, 255, 255], dtype=np.float32)
    return np.median(pixels.reshape(-1, 3), axis=0)

def estimate_background(image):
    h, w, _ = image.shape
    dh = max(2, int(h * 0.05))
    dw = max(2, int(w * 0.05))
    corners = [
        image[0:dh, 0:dw],
        image[0:dh, w - dw:w],
        image[h - dh:h, 0:dw],
        image[h - dh:h, w - dw:w],
    ]
    return _median_color(np.vstack([c.reshape(-1, 3) for c in corners]))

def sample_line_color(image, x, y, bg_color, patch_radius=3):
    h, w, _ = image.shape
    xi, yi = int(round(x)), int(round(y))
    x0 = max(0, xi - patch_radius)
    x1 = min(w, xi + patch_radius + 1)
    y0 = max(0, yi - patch_radius)
    y1 = min(h, yi + patch_radius + 1)
    patch = image[y0:y1, x0:x1].reshape(-1, 3)
    if patch.size == 0:
        return np.array([0, 0, 0], dtype=np.float32)

    dist_bg = np.linalg.norm(patch.astype(np.float32) - bg_color, axis=1)
    non_bg = patch[dist_bg > BG_TOL]
    if len(non_bg) == 0:
        return _median_color(patch)
    return _median_color(non_bg)

def _find_endpoint(image, x, y_start, direction, target_color, bg_color, tol, bg_tol, gap_limit, window, min_hits):
    height, width, _ = image.shape
    x = int(round(x))
    x_min = max(0, x - window)
    x_max = min(width - 1, x + window)

    last_match = None
    gap = 0
    y = int(round(y_start))
    while 0 <= y < height:
        strip = image[y, x_min : x_max + 1]
        strip_f = strip.astype(np.float32)
        dist = np.linalg.norm(strip_f - target_color, axis=1)
        dist_bg = np.linalg.norm(strip_f - bg_color, axis=1)
        hits = np.sum((dist <= tol) & (dist_bg >= bg_tol))
        if hits >= min_hits:
            last_match = y
            gap = 0
        else:
            if last_match is not None:
                gap += 1
                if gap >= gap_limit:
                    break
        y += direction
    return last_match

def detect_error_bars(image, line_groups, tol=38.0, bg_tol=18.0, gap_limit=3, window=2, min_hits=1):
    bg_color = estimate_background(image)
    output = []
    for line in line_groups:
        line_name = line.get("lineName", "")
        points_output = []
        for point in line.get("points", []):
            x, y = point["x"], point["y"]
            target = sample_line_color(image, x, y, bg_color)

            up = _find_endpoint(image, x, y, -1, target, bg_color, tol, bg_tol, gap_limit, window, min_hits)
            down = _find_endpoint(image, x, y, 1, target, bg_color, tol, bg_tol, gap_limit, window, min_hits)

            if up is None:
                up = y
            if down is None:
                down = y

            points_output.append({
                "data_point": {"x": float(x), "y": float(y)},
                "upper_error_bar": {"x": float(x), "y": float(up)},
                "lower_error_bar": {"x": float(x), "y": float(down)},
            })
        output.append({"lineName": line_name, "points": points_output})
    return output

In [5]:
# Run prediction for all input.json files
input_files = sorted(INPUT_DIR.glob("*.json"))
print(f"Found {len(input_files)} input files")

for idx, input_path in enumerate(input_files, start=1):
    data = json.loads(input_path.read_text(encoding="utf-8"))
    image_file = data.get("image_file")
    line_groups = data.get("data_points", [])

    img_path = IMAGES_DIR / image_file
    if not img_path.exists():
        print(f"[skip] image not found for {input_path.name}")
        continue

    image = np.array(Image.open(img_path).convert("RGB"))
    output = {
        "image_file": image_file,
        "error_bars": detect_error_bars(
            image,
            line_groups,
            tol=COLOR_TOL,
            bg_tol=BG_TOL,
            gap_limit=GAP_LIMIT,
            window=WINDOW,
            min_hits=MIN_HITS,
        ),
    }

    (PRED_DIR / f"{input_path.stem}.json").write_text(json.dumps(output, indent=2), encoding="utf-8")

    if idx % 50 == 0:
        print(f"Predicted {idx}/{len(input_files)}")

print("Prediction done.")

Found 150 input files
Predicted 50/150
Predicted 100/150
Predicted 150/150
Prediction done.


In [6]:
# Evaluation settings
PIX_TOL = 3.0   # pixel tolerance for correct detection

def _point_key(line_name, pt):
    return (
        line_name,
        round(float(pt["data_point"]["x"]), 2),
        round(float(pt["data_point"]["y"]), 2),
    )

def index_points(error_bars):
    index = {}
    for line in error_bars:
        line_name = line.get("lineName", "")
        for pt in line.get("points", []):
            key = _point_key(line_name, pt)
            index[key] = pt
    return index

def within_tol(a, b, tol):
    return abs(a - b) <= tol

tp = 0
fp = 0
fn = 0
abs_err_up = []
abs_err_down = []

gt_files = sorted(GT_DIR.glob("*.json"))
for gt_path in gt_files:
    pred_path = PRED_DIR / gt_path.name
    if not pred_path.exists():
        # no predictions for this image
        gt = json.loads(gt_path.read_text(encoding="utf-8"))
        gt_index = index_points(gt.get("error_bars", []))
        fn += len(gt_index)
        continue

    gt = json.loads(gt_path.read_text(encoding="utf-8"))
    pred = json.loads(pred_path.read_text(encoding="utf-8"))

    gt_index = index_points(gt.get("error_bars", []))
    pred_index = index_points(pred.get("error_bars", []))

    matched_pred_keys = set()
    for key, g in gt_index.items():
        p = pred_index.get(key)
        if p is None:
            fn += 1
            continue

        matched_pred_keys.add(key)

        gu = float(g["upper_error_bar"]["y"])
        gl = float(g["lower_error_bar"]["y"])
        pu = float(p["upper_error_bar"]["y"])
        pl = float(p["lower_error_bar"]["y"])

        abs_err_up.append(abs(gu - pu))
        abs_err_down.append(abs(gl - pl))

        ok = within_tol(gu, pu, PIX_TOL) and within_tol(gl, pl, PIX_TOL)
        if ok:
            tp += 1
        else:
            fp += 1

    # extra predictions not matched to any gt
    extra = len(pred_index) - len(matched_pred_keys)
    if extra > 0:
        fp += extra

total = tp + fp + fn
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
accuracy = tp / total if total else 0.0

mae_up = float(np.mean(abs_err_up)) if abs_err_up else 0.0
mae_down = float(np.mean(abs_err_down)) if abs_err_down else 0.0

# Confusion matrix (per point): [[TP, FP], [FN, TN]]
confusion_matrix = [[tp, fp], [fn, 0]]

{
    "points_total": total,
    "points_correct_within_tol": tp,
    "points_incorrect": fp,
    "points_missing": fn,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "mae_upper": mae_up,
    "mae_lower": mae_down,
    "pixel_tolerance": PIX_TOL,
    "confusion_matrix": confusion_matrix,
}

{'points_total': 4839,
 'points_correct_within_tol': 1115,
 'points_incorrect': 3724,
 'points_missing': 0,
 'accuracy': 0.23041950816284357,
 'precision': 0.23041950816284357,
 'recall': 1.0,
 'f1': 0.3745381256298287,
 'mae_upper': 24.454727816987763,
 'mae_lower': 26.000820206633026,
 'pixel_tolerance': 3.0,
 'confusion_matrix': [[1115, 3724], [0, 0]]}